# Tutorial: Temporal analysis

This tutorial demonstrates temporal utilities for STREAM observations: observation-period summaries, common periods across datasets, water years, meteorological seasons, and daily or monthly resampling.

## 1. Connect and import the temporal utilities

In [ ]:
import pandas as pd

from utils.client import StreamClient
from utils.joins import join_timeseries
from utils.temporal import (
    add_water_year,
    common_observation_period,
    filter_season,
    resample_timeseries,
    summarize_observation_period,
)

stream = StreamClient()

## 2. Retrieve manageable hourly subsets

Use one site and a three-year window so the tutorial remains responsive. Water quality and streamflow are requested separately because their actual periods of available observations may differ.

In [ ]:
example_site = "STREAM-gauge-1022"
requested_start = "2019-01-01"
requested_end = "2021-12-31 23:59:59"

In [ ]:
water_quality = stream.read_water_quality(
    site_ids=example_site,
    variables="WTemp_C",
    start=requested_start,
    end=requested_end,
    quality=["approved", "provisional"],
)

streamflow = stream.read_streamflow(
    site_ids=example_site,
    start=requested_start,
    end=requested_end,
    quality=["approved", "provisional"],
)

print(f"Water-quality rows: {len(water_quality):,}")
print(f"Streamflow rows: {len(streamflow):,}")

## 3. Summarize observation periods

The summary reports valid timestamp counts and the span from the first to last timestamp for each site. `duration_days` describes the span; it does not measure completeness or guarantee that every hour is present.

In [ ]:
water_quality_period = summarize_observation_period(water_quality).assign(
    dataset="water_quality"
)
streamflow_period = summarize_observation_period(streamflow).assign(
    dataset="streamflow"
)

period_summary = pd.concat(
    [water_quality_period, streamflow_period],
    ignore_index=True,
)
period_summary[[
    "dataset", "STREAM_ID", "observation_count",
    "start_time", "end_time", "duration_days",
]]

## 4. Find and select the common period

The common period begins at the later start time and ends at the earlier end time. Using it prevents an empty or misleading time-based merge caused by non-overlapping selections.

In [ ]:
common_start, common_end = common_observation_period(
    water_quality,
    streamflow,
)

print("Common start:", common_start)
print("Common end:  ", common_end)

In [ ]:
water_quality_common = water_quality[
    water_quality["DateTime"].between(common_start, common_end)
].copy()
streamflow_common = streamflow[
    streamflow["DateTime"].between(common_start, common_end)
].copy()

print(f"Water-quality rows in common period: {len(water_quality_common):,}")
print(f"Streamflow rows in common period: {len(streamflow_common):,}")

## 5. Add water years

The default U.S. water year begins October 1 and is labeled by the calendar year in which it ends.

In [ ]:
water_quality_with_year = add_water_year(water_quality_common)

water_year_summary = (
    water_quality_with_year
    .groupby(["STREAM_ID", "water_year"], as_index=False)
    .agg(
        mean_temperature_C=("WTemp_C", "mean"),
        observation_count=("WTemp_C", "count"),
    )
)
water_year_summary

## 6. Compare meteorological seasons

STREAM supports the four standard meteorological seasons: winter, spring, summer, and fall.

In [ ]:
season_summaries = []
for season in ["winter", "spring", "summer", "fall"]:
    seasonal_data = filter_season(water_quality_common, season)
    season_summaries.append(
        {
            "season": season,
            "mean_temperature_C": seasonal_data["WTemp_C"].mean(),
            "observation_count": seasonal_data["WTemp_C"].count(),
        }
    )

season_summary = pd.DataFrame(season_summaries)
season_summary

## 7. Calculate daily summaries

Only explicitly selected numeric columns are aggregated. Quality flags are intentionally excluded.

In [ ]:
daily_temperature = resample_timeseries(
    water_quality_common,
    value_columns="WTemp_C",
    frequency="daily",
    statistic="mean",
)

daily_streamflow = resample_timeseries(
    streamflow_common,
    value_columns="Q_m3s",
    frequency="daily",
    statistic="mean",
)

daily_temperature.head()

## 8. Calculate monthly summaries

In [ ]:
monthly_temperature = resample_timeseries(
    water_quality_common,
    value_columns="WTemp_C",
    frequency="monthly",
    statistic="mean",
)

monthly_streamflow = resample_timeseries(
    streamflow_common,
    value_columns="Q_m3s",
    frequency="monthly",
    statistic="median",
)

monthly_temperature.head()

## 9. Join summaries with matching temporal resolution

After both datasets are resampled monthly, they can be joined on exact station and month values. `join_timeseries()` uses pandas for the merge but adds safeguards: it rejects missing or duplicate station/timestamp keys and clearly distinguishes non-overlapping periods from periods with no exact matches. This prevents an unnoticed many-to-many merge from inflating the result.

In [ ]:
monthly_combined = join_timeseries(
    monthly_temperature,
    monthly_streamflow,
    suffixes=("_temperature", "_flow"),
)

monthly_combined.head()

## Interpretation notes

- A first-to-last observation span is not a completeness measurement; gaps may exist inside it.
- Choose aggregation statistics that match the variable and research question. For example, mean temperature and total precipitation have different interpretations.
- Resampling does not create observations. Intervals with no source values remain missing.
- Review source quality flags before resampling; this tutorial filters to approved and provisional records during retrieval.